[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/jvm-internals/blob/main/notebooks/O01/lesson.ipynb)

**How big is a bare Object, and why is the answer different from the one you learned?**

Run the cell below first. Everything after it assumes `jvx` is loaded, and the numbers on this page came from `jdk-27+35`.

Every Java object costs more than the fields you declared. The extra is the header, and for about twenty years the number everybody learned was 16 bytes for an empty object: 8 for the mark word, 4 for the class pointer, 4 for alignment.

That number is wrong on the JVM you are about to run. It changed in JDK 27, by default, for everybody, and it changed by enough that a heap full of small objects can shrink by a fifth.

The interesting part is not the new number. It is that knowing the new number still will not let you predict the size of an `Integer`, and there is a good reason for that which almost nobody thinks about until they measure it.

You are going to measure it. Three questions, and you have to commit to an answer before each one.

In [ ]:
// Generated by tools/build.py. Do not edit this cell, it is overwritten on
// every build. Edit the sources and run `python tools/build.py notebooks`.
//
// helper surface   jvx/00-imports.jsh, jvx/05-ui.jsh, jvx/10-markword.jsh, jvx/15-gate.jsh, jvx/18-lens.jsh, jvx/20-jvx.jsh
// bit positions    docs/generated/markword.json, read by tools/gen_markword.py
//                  from src/hotspot/share/oops/markWord.hpp at jdk-27+35
//
// This cell assumes a Java kernel is already running. Installing the pinned
// JDK from a cold Colab runtime is a separate step that is still being
// measured, and it goes here when it is. See issue #1.

// JShell imports a useful default set, but not these. They are separate snippets on
// purpose: an import in JShell applies to everything typed afterwards, so putting
// them first means a reader's own cells get them too without asking.
//
// The second group is here for a reason worth knowing about, because it cost an
// afternoon. A jshell you start in a terminal imports java.nio.file.* for you. The
// notebook kernel does not: JJava sets its own list, which is java.util, java.io,
// java.math, java.net, java.time, java.util.concurrent, java.util.prefs and
// java.util.regex, and nothing else. So a helper surface that loads perfectly in a
// terminal can fail to compile in the kernel every reader uses, and the error a reader
// sees is `cannot find symbol: variable jvx`, which points at the wrong thing entirely.
// tools/test_jvx_ui.py loads the whole surface with only the kernel's imports for that
// reason.
import java.lang.management.ManagementFactory;
import java.lang.management.RuntimeMXBean;
import java.lang.reflect.Field;
import java.lang.reflect.Method;
import java.lang.reflect.Modifier;
import com.sun.management.HotSpotDiagnosticMXBean;
import com.sun.management.VMOption;

import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;

// Everything a lesson draws on the screen goes through here, and the shape of it comes
// straight out of a measurement rather than out of taste.
//
// probes/widgets measured twelve ways of getting something in front of a reader, in four
// places each, and a saved notebook that nobody has run keeps four of them: a style
// attribute on the element, details and summary, an img whose src is an SVG data URI, and
// markdown. Everything else is sanitized away. Style tags are removed while the class
// attribute is kept, so the rule is gone and the hook that wanted it is still there. An id
// is renamed to data-jupyter-id, so every selector quietly stops matching. Form controls
// arrive disabled. Scripts, onclick and iframes are removed outright.
//
// That is the state a reader is in when they click a link and read the page, which is most
// readers most of the time. So this file uses those four things and nothing else. There is
// no style tag, no id, no script and no input anywhere below, and tools/test_jvx_ui.py
// fails the build if one appears. The full reading is in docs/probes/widgets.md.

class Ui {

    static final String FONT = "system-ui, -apple-system, Segoe UI, Roboto, sans-serif";
    static final String MONO = "ui-monospace, SFMono-Regular, Menlo, Consolas, monospace";
    static final String INK = "#212529";
    static final String MUTED = "#868e96";
    static final String BLUE = "#4c6ef5";
    static final String GREEN = "#2f9e44";
    static final String ORANGE = "#e8590c";

    // -- getting a payload to the front end ----------------------------------------
    //
    // The kernel is JJava, and the `display` it puts in scope is a static method on
    // org.dflib.jjava.jupyter.kernel.BaseNotebookStatics. Calling it by name would work
    // in a notebook and would break everywhere else, because the same helper surface
    // gets piped into a plain jshell when somebody debugs the bootstrap, and JShell
    // refuses to run a method whose body names something that does not exist. Looking it
    // up reflectively answers both questions at once: whether there is a screen to draw
    // on, and how to draw on it.

    private static Method displayMethod;
    private static boolean lookedForIt = false;

    /** Is there a front end here that can render markup, or are we in a terminal. */
    static boolean rich() {
        if (!lookedForIt) {
            lookedForIt = true;
            try {
                Class<?> statics =
                    Class.forName("org.dflib.jjava.jupyter.kernel.BaseNotebookStatics");
                displayMethod = statics.getMethod("display", Object.class, String[].class);
            } catch (Throwable notANotebook) {
                displayMethod = null;
            }
        }
        return displayMethod != null;
    }

    /**
     * Put markup on the screen. False means there was no screen, so print text instead.
     *
     * The return value of `display` is thrown away on purpose and this is the only place
     * in the project that calls it. `display` hands back the id it assigned, JShell
     * prints the value of the last expression it evaluates, and the result is a line of
     * hex under every widget on the page. Measured on twelve cells out of twelve. One
     * assignment here is the whole fix.
     */
    static boolean html(String markup) {
        if (!rich()) return false;
        try {
            String ignored = (String) displayMethod.invoke(
                null, markup, new String[] { "text/html" });
            return true;
        } catch (Throwable t) {
            // A front end that turned out not to want it is not worth an exception in a
            // reader's face. Saying false sends the caller to the text version.
            return false;
        }
    }

    // -- building the markup ---------------------------------------------------------

    static String esc(String text) {
        return text.replace("&", "&amp;")
                   .replace("<", "&lt;")
                   .replace(">", "&gt;")
                   .replace("\"", "&quot;");
    }

    /** Escaped text, with `backticks` turned into code spans, which lessons already write. */
    static String prose(String text) {
        String[] parts = esc(text).split("`", -1);
        StringBuilder out = new StringBuilder();
        for (int i = 0; i < parts.length; i++) {
            // The odd numbered pieces are the ones between a pair of backticks. An
            // unclosed backtick leaves the last piece odd, and that one gets its backtick
            // put back and goes out as plain text, because a lesson with a typo in it
            // should look wrong on the page rather than quietly lose a character.
            if (i % 2 == 1 && i < parts.length - 1) {
                out.append(code(parts[i]));
            } else if (i % 2 == 1) {
                out.append("`").append(parts[i]);
            } else {
                out.append(parts[i]);
            }
        }
        return out.toString();
    }

    /** Already escaped text in a code span. Use prose() for anything a lesson typed. */
    static String code(String escaped) {
        return "<code style=\"font-family:" + MONO + ";font-size:0.92em;background:#e9ecef;"
            + "padding:1px 5px;border-radius:3px\">" + escaped + "</code>";
    }

    static String card(String accent, String label, String body) {
        return "<div style=\"font-family:" + FONT + ";color:" + INK + ";max-width:46em;"
            + "border:1px solid #dee2e6;border-left:5px solid " + accent + ";"
            + "border-radius:6px;background:#f8f9fa;padding:14px 16px;margin:4px 0\">"
            + "<div style=\"font-size:11px;font-weight:700;letter-spacing:0.08em;"
            + "text-transform:uppercase;color:" + accent + ";margin-bottom:8px\">"
            + esc(label) + "</div>"
            + body
            + "</div>";
    }

    /**
     * The one interactive element that survives everywhere.
     *
     * No CSS and no JavaScript, so there is nothing for a sanitizer to take away. Open it
     * when the reader has earned what is inside and leave it shut when they have not.
     */
    static String details(String summary, String body, boolean open) {
        return "<details" + (open ? " open" : "") + " style=\"margin-top:10px\">"
            + "<summary style=\"cursor:pointer;font-weight:600;color:" + BLUE + "\">"
            + esc(summary) + "</summary>"
            + "<div style=\"margin-top:8px\">" + body + "</div></details>";
    }

    /**
     * A picture, as an img with the SVG base64 encoded into the src.
     *
     * This is the useful half of the whole widget probe. An `image/svg+xml` output is
     * shown as escaped source text in a notebook nobody has run, which is worse than
     * showing nothing, and the identical bytes inside an img render in every environment
     * measured. So anything this project can draw as an SVG, it can put on any page.
     */
    static String img(String svg, String alt) {
        String encoded = Base64.getEncoder().encodeToString(svg.getBytes(StandardCharsets.UTF_8));
        return "<img alt=\"" + esc(alt) + "\" style=\"max-width:100%;display:block;"
            + "margin:4px 0\" src=\"data:image/svg+xml;base64," + encoded + "\">";
    }

    static String line(String body) {
        return "<div style=\"margin:4px 0;line-height:1.5\">" + body + "</div>";
    }

    static String small(String body) {
        return "<div style=\"margin-top:10px;font-size:13px;color:" + MUTED
            + ";line-height:1.5\">" + body + "</div>";
    }
}

// The mark word is the first eight bytes of every object on the heap, and it carries
// several unrelated things at once: the lock state, the identity hash, the age the
// collector uses to decide about promotion, and on JDK 27 the class pointer as well.
//
// None of the numbers below are typed by hand. The placeholder line further down is
// replaced by tools/build.py using docs/generated/markword.json, which
// tools/gen_markword.py works out from HotSpot's own markWord.hpp at the pinned tag.
// The citation on each field is the line of markWord.hpp it came from, so a reader
// who does not believe a number can go and look at the line that produced it.

class MarkWord {

    static final class Field {
        final String name;
        final int shift;
        final int bits;
        final String meaning;
        final String citation;

        Field(String name, int shift, int bits, String meaning, String citation) {
            this.name = name;
            this.shift = shift;
            this.bits = bits;
            this.meaning = meaning;
            this.citation = citation;
        }

        /** The value of this field in the given word. */
        long of(long word) {
            return (word >>> shift) & ((1L << bits) - 1);
        }

        /** The bit positions this field occupies, high end first, the way a diagram reads. */
        String span() {
            return (shift + bits - 1) + ".." + shift;
        }
    }

    static final String SOURCE_PATH = "src/hotspot/share/oops/markWord.hpp";
    static final String SOURCE_TAG = "jdk-27+35";
    static final String SOURCE_SHA256 = "c249a091cf7dcae455d073a32f29d65b0bf2c1f0b452bc5705b706a6f1aba267";

    static final Field[] FIELDS = {
        new Field("lock", 0, 2,
            "the lock state, and whether a collector has marked or forwarded the object",
            "src/hotspot/share/oops/markWord.hpp:124@jdk-27+35"),
        new Field("self_fwd", 2, 1,
            "set when a collector forwarded the object in place",
            "src/hotspot/share/oops/markWord.hpp:125@jdk-27+35"),
        new Field("age", 3, 4,
            "how many collections the object has survived",
            "src/hotspot/share/oops/markWord.hpp:126@jdk-27+35"),
        new Field("valhalla", 7, 4,
            "reserved for Valhalla, unused today",
            "src/hotspot/share/oops/markWord.hpp:127@jdk-27+35"),
        new Field("hash", 11, 31,
            "the identity hash, written the first time anything asks for it",
            "src/hotspot/share/oops/markWord.hpp:128@jdk-27+35"),
        new Field("klass", 42, 22,
            "the compressed class pointer, present only when UseCompactObjectHeaders is on",
            "src/hotspot/share/oops/markWord.hpp:150@jdk-27+35"),
    };

    static final String[] LOCK_BITS = {"00", "01", "10", "11"};
    static final String[] LOCK_MEANING = {
        "locked, a stack lock is held, and the real header is in the lock record",
        "unlocked, the ordinary state",
        "monitor, the lock is inflated",
        "marked, a collector is using the word, and the real header is elsewhere",
    };

    static Field field(String name) {
        for (Field f : FIELDS) {
            if (f.name.equals(name)) return f;
        }
        throw new IllegalArgumentException(
            "no field called " + name + " in the mark word at " + SOURCE_TAG);
    }

    static long get(long word, String name) {
        return field(name).of(word);
    }

    static String hex(long word) {
        return String.format("0x%016x", word);
    }

    /** All 64 bits, grouped in bytes, so the reader can count them. */
    static String bin(long word) {
        StringBuilder b = new StringBuilder(72);
        for (int i = 63; i >= 0; i--) {
            b.append((word >>> i) & 1L);
            if (i % 8 == 0 && i != 0) b.append(' ');
        }
        return b.toString();
    }

    /**
     * The same 64 bits with each field's own bits shown and everything else dimmed to
     * a dot. Reading a hex number and believing where the boundaries are is exactly
     * the step where people go wrong, so this draws the boundaries.
     */
    static String ruler(long word) {
        StringBuilder out = new StringBuilder();
        for (int i = FIELDS.length - 1; i >= 0; i--) {
            Field f = FIELDS[i];
            StringBuilder row = new StringBuilder(72);
            for (int bit = 63; bit >= 0; bit--) {
                boolean mine = bit >= f.shift && bit < f.shift + f.bits;
                row.append(mine ? Character.forDigit((int) ((word >>> bit) & 1L), 10) : '.');
                if (bit % 8 == 0 && bit != 0) row.append(' ');
            }
            out.append(row).append("  ").append(f.name).append('\n');
        }
        return out.toString();
    }

    static String decode(long word) {
        StringBuilder b = new StringBuilder();
        b.append(hex(word)).append('\n');
        b.append(bin(word)).append('\n');
        b.append('\n');
        b.append(ruler(word));
        b.append('\n');
        b.append(String.format("%-8s %-9s %-12s %s%n", "bits", "field", "value", "meaning"));
        b.append("-".repeat(78)).append('\n');
        for (int i = FIELDS.length - 1; i >= 0; i--) {
            Field f = FIELDS[i];
            b.append(String.format("%-8s %-9s %-12s %s%n",
                f.span(), f.name, "0x" + Long.toHexString(f.of(word)), f.meaning));
        }
        b.append('\n');
        int lock = (int) get(word, "lock");
        b.append("lock ").append(LOCK_BITS[lock]).append(", ").append(LOCK_MEANING[lock]).append('\n');
        return b.toString();
    }

    /** Where every number above came from, for a reader who wants to check one. */
    static String provenance() {
        StringBuilder b = new StringBuilder();
        b.append("generated from ").append(SOURCE_PATH).append(" at ").append(SOURCE_TAG).append('\n');
        b.append("sha256 ").append(SOURCE_SHA256).append('\n');
        b.append('\n');
        for (int i = FIELDS.length - 1; i >= 0; i--) {
            Field f = FIELDS[i];
            b.append(String.format("%-9s %s%n", f.name, f.citation));
        }
        return b.toString();
    }
}

// A prediction gate.
//
// The rule this project runs on is that a reader who has not committed to an answer
// does not really read the reveal. They skim it and come away feeling like they knew.
// Committing to a wrong answer first is what makes the correction stick, so the gate
// makes you write one down and does not show you anything until you have.
//
// There are two renderings and the text one is not a fallback in the apologetic sense.
// It works in a terminal, in a printed transcript and in any notebook with no HTML, and
// every word of it is in the card version too. What the card adds is a shape the eye can
// find on a long page, and one thing the text cannot do: on a page nobody has run, the
// answer sits inside a details element, so a reader scrolling past a reveal has to decide
// to open it rather than have it handed to them. That is the gate working in the one
// environment where it used to be impossible.
//
// A lesson never names Gate. It calls jvx.gate, jvx.answer and jvx.reveal, which is why
// this file could change shape twice without touching a lesson.

class Gate {

    static final Map<String, String> question = new LinkedHashMap<>();
    static final Map<String, String> answered = new LinkedHashMap<>();

    static void ask(String id, String text, String... options) {
        question.put(id, text);
        if (Ui.html(askHtml(id, text, options))) return;

        System.out.println(text);
        System.out.println();
        for (String option : options) {
            System.out.println("    " + option);
        }
        System.out.println();
        System.out.println("Pick one before you run anything else. There is a right answer and");
        System.out.println("the wrong ones are wrong for reasons worth knowing.");
        System.out.println();
        System.out.println("    jvx.answer(\"" + id + "\", \"a\")");
    }

    static void answer(String id, String choice) {
        if (!question.containsKey(id)) {
            String missing = "No gate called " + id + " is open. Run the gate cell above first.";
            if (!Ui.html(Ui.card(Ui.ORANGE, "nothing to answer", Ui.line(Ui.esc(missing))))) {
                System.out.println(missing);
            }
            return;
        }
        String cleaned = choice.trim().toLowerCase();
        answered.put(id, cleaned);
        if (Ui.html(answerHtml(id, cleaned))) return;
        System.out.println("Recorded " + cleaned + " for " + id + ". Now run the next cell.");
    }

    static void reveal(String id, String correct) {
        String expected = correct.trim().toLowerCase();
        String mine = answered.get(id);
        if (Ui.html(revealHtml(id, expected, mine))) return;

        if (mine == null) {
            // Not a refusal. Somebody who hit Run All should not end up staring at a
            // cell that will not talk to them. But it says what was lost, because it
            // was a real thing and not a formality.
            System.out.println("You did not commit to an answer, so this is worth less to you");
            System.out.println("than it would have been. The answer is " + expected + ".");
            return;
        }
        if (mine.equals(expected)) {
            System.out.println("You said " + mine + ", and that is right.");
            System.out.println("Read on anyway. Being right for the wrong reason is common here.");
        } else {
            System.out.println("You said " + mine + ". The answer is " + expected + ".");
            System.out.println("This is the useful outcome. Read what follows carefully.");
        }
    }

    // -- the card version ------------------------------------------------------------
    //
    // These three build markup and put nothing on the screen, which is what makes them
    // testable without a kernel. tools/test_jvx_ui.py runs them in a plain jshell and
    // checks both what they say and that they use only the four things that survive a
    // notebook nobody has run.

    static String askHtml(String id, String text, String[] options) {
        StringBuilder body = new StringBuilder();
        body.append("<div style=\"font-size:15px;line-height:1.5;margin-bottom:10px\">")
            .append(Ui.prose(text))
            .append("</div>");
        for (String option : options) {
            body.append(optionHtml(option));
        }
        body.append(Ui.small(
            "Pick one before you run anything else. There is a right answer and the wrong "
            + "ones are wrong for reasons worth knowing. Put yours in the next cell: "
            + Ui.code("jvx.answer(&quot;" + Ui.esc(id) + "&quot;, &quot;a&quot;)")));
        return Ui.card(Ui.BLUE, "predict", body.toString());
    }

    /**
     * One option, with its letter pulled out into a chip.
     *
     * Lessons write options as "a) 8", so the letter is already there and splitting it
     * out is presentation. An option written some other way is printed whole rather than
     * mangled into the shape this method was hoping for.
     */
    static String optionHtml(String option) {
        String letter = "";
        String rest = option;
        int bracket = option.indexOf(')');
        if (bracket == 1 && Character.isLetter(option.charAt(0))) {
            letter = option.substring(0, 1);
            rest = option.substring(2).trim();
        }
        String chip = letter.isEmpty() ? "" :
            "<span style=\"font-family:" + Ui.MONO + ";font-weight:700;color:" + Ui.BLUE
            + ";margin-right:10px\">" + Ui.esc(letter) + "</span>";
        return "<div style=\"margin:5px 0 5px 4px;line-height:1.5\">"
            + chip + Ui.prose(rest) + "</div>";
    }

    static String answerHtml(String id, String choice) {
        return Ui.card(Ui.MUTED, "recorded",
            Ui.line("You said " + Ui.code(Ui.esc(choice)) + " for "
                + Ui.code(Ui.esc(id)) + ". It is not marked yet. Run the next cell."));
    }

    static String revealHtml(String id, String expected, String mine) {
        if (mine == null) {
            // Nobody ran the answer cell, which is also what a reader of the published
            // page is looking at, since nothing on it has been run at all. Putting the
            // answer behind a details is the whole reason this rendering exists: it is
            // the only interaction that survives an unrun notebook, so it is the only way
            // a gate can still be a gate on a page somebody is only reading.
            return Ui.card(Ui.MUTED, "not answered",
                Ui.line("You did not commit to an answer, so this is worth less to you than "
                    + "it would have been. Write one down and it will mark it.")
                + Ui.details("Show me the answer anyway",
                    Ui.line("The answer is " + Ui.code(Ui.esc(expected)) + "."), false));
        }
        if (mine.equals(expected)) {
            return Ui.card(Ui.GREEN, "right",
                Ui.line("You said " + Ui.code(Ui.esc(mine)) + ", and that is right.")
                + Ui.small("Read on anyway. Being right for the wrong reason is common here."));
        }
        return Ui.card(Ui.ORANGE, "worth having",
            Ui.line("You said " + Ui.code(Ui.esc(mine)) + ". The answer is "
                + Ui.code(Ui.esc(expected)) + ".")
            + Ui.small("This is the useful outcome. Read what follows carefully."));
    }
}

// HeapLens: what one object looks like in memory, byte by byte.
//
// This file draws and measures nothing. It is handed a list of slots that somebody else
// measured and turns them into a picture, which is what makes it testable without a JVM
// to point at: the test can hand it a layout it invented and check the drawing, and
// separately check that the measuring produces the layout it should.
//
// The picture is an SVG inside an img with a data URI, because probes/widgets measured
// that as the only kind of picture that renders in all four places a reader might be,
// including a saved notebook nobody has run. Inside the img the SVG is never sanitized,
// so the drawing can use anything SVG has.

class Lens {

    /**
     * One run of bytes in an object, and what is living there.
     *
     * `kind` is what it is rather than what it looks like: header, field, gap or padding.
     * The difference between a gap and padding matters and is the whole reason a reader
     * is looking at this. A gap is alignment inside the object, put there because the
     * next field could not start where the last one ended. Padding is at the end, put
     * there because the whole object has to be a multiple of the alignment. One is the
     * field order's fault and can be fixed by reordering. The other cannot.
     */
    record Slot(String label, long offset, long width, String kind) {}

    static final int CELL = 34;      // one byte
    static final int ROW = 42;       // one 8 byte word
    static final int GUTTER = 46;    // the offset down the left
    static final int TOP = 28;       // the byte ruler across the top
    static final int PAD = 12;

    static String colour(String kind, int index) {
        if (kind.equals("header")) return "#4c6ef5";
        if (kind.equals("padding")) return "#adb5bd";
        if (kind.equals("gap")) return "#ffa94d";
        // Fields cycle, so two neighbours never share a colour and a reader can see
        // where one stops without reading the label.
        String[] wheel = { "#2f9e44", "#1098ad", "#7048e8", "#e64980", "#f08c00" };
        return wheel[Math.floorMod(index, wheel.length)];
    }

    static String ink(String kind) {
        return kind.equals("padding") ? "#495057" : "#ffffff";
    }

    /** The picture. One row per eight byte word, one rect per slot per row. */
    static String svg(List<Slot> slots, long size) {
        long rows = (size + 7) / 8;
        int width = PAD * 2 + GUTTER + 8 * CELL;
        int height = (int) (TOP + rows * ROW + PAD);

        StringBuilder out = new StringBuilder();
        out.append("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"").append(width)
           .append("\" height=\"").append(height)
           .append("\" viewBox=\"0 0 ").append(width).append(" ").append(height)
           .append("\" font-family=\"ui-monospace, SFMono-Regular, Menlo, monospace\">");
        out.append("<rect width=\"").append(width).append("\" height=\"").append(height)
           .append("\" fill=\"#ffffff\"/>");

        // The byte ruler. Which byte of the word, not which byte of the object, because
        // the offset down the side already says that and saying it twice is noise.
        for (int b = 0; b < 8; b++) {
            out.append(text(PAD + GUTTER + b * CELL + CELL / 2, TOP - 10, String.valueOf(b),
                11, "#adb5bd", "middle", false));
        }

        for (long row = 0; row < rows; row++) {
            int y = (int) (TOP + row * ROW);
            out.append(text(PAD + GUTTER - 12, y + ROW / 2 + 4, String.valueOf(row * 8),
                12, "#868e96", "end", false));
        }

        int fieldIndex = 0;
        for (Slot slot : slots) {
            int index = slot.kind().equals("field") ? fieldIndex++ : 0;
            long from = slot.offset();
            long to = slot.offset() + slot.width();
            // A slot that crosses a word boundary is drawn once per row it touches, so
            // the picture stays a grid and a long field still reads as one thing.
            for (long start = from; start < to; ) {
                long row = start / 8;
                long end = Math.min(to, (row + 1) * 8);
                int x = (int) (PAD + GUTTER + (start % 8) * CELL);
                int y = (int) (TOP + row * ROW);
                int w = (int) ((end - start) * CELL) - 3;
                out.append("<rect x=\"").append(x + 1).append("\" y=\"").append(y + 3)
                   .append("\" width=\"").append(w).append("\" height=\"").append(ROW - 9)
                   .append("\" rx=\"4\" fill=\"").append(colour(slot.kind(), index))
                   .append(slot.kind().equals("padding")
                       ? "\" fill-opacity=\"0.35\" stroke=\"#adb5bd\" stroke-dasharray=\"3 3\"/>"
                       : "\"/>");
                // A label needs room. Two bytes is not room, and a clipped word is worse
                // than no word, because the offset in the details below is exact anyway.
                if (w >= 62) {
                    out.append(text(x + 1 + w / 2, y + ROW / 2 + 1, slot.label(), 13,
                        ink(slot.kind()), "middle", true));
                }
                start = end;
            }
        }
        out.append("</svg>");
        return out.toString();
    }

    static String text(int x, int y, String body, int size, String fill, String anchor,
                       boolean bold) {
        return "<text x=\"" + x + "\" y=\"" + y + "\" font-size=\"" + size + "\" fill=\"" + fill
            + "\" text-anchor=\"" + anchor + "\""
            + (bold ? " font-weight=\"600\"" : "") + ">" + Ui.esc(body) + "</text>";
    }

    /**
     * What the picture says, for somebody who cannot see it.
     *
     * Not an afterthought and not generated from the same string twice. A screen reader
     * gets this, and so does anyone whose front end blocks images, and it is the same
     * sentence the text version prints in a terminal.
     */
    static String alt(String title, List<Slot> slots, long size) {
        StringBuilder out = new StringBuilder("the layout of " + title + ", " + size + " bytes: ");
        for (int i = 0; i < slots.size(); i++) {
            Slot s = slots.get(i);
            if (i > 0) out.append(", ");
            out.append(s.label()).append(" at ").append(s.offset())
               .append(" for ").append(s.width()).append(s.width() == 1 ? " byte" : " bytes");
        }
        return out.toString();
    }

    /** The card: the picture, then the numbers behind it one click away. */
    static String card(String title, List<Slot> slots, long size, String note) {
        StringBuilder rows = new StringBuilder();
        int fieldIndex = 0;
        for (Slot s : slots) {
            // Counted the same way the picture counts, so the dot beside a name is the
            // colour of the box it points at. Two lists that drift apart are worse than
            // one list, so there is exactly one rule and both of them use it.
            int index = s.kind().equals("field") ? fieldIndex++ : 0;
            // white-space:pre, or the alignment in `row` does nothing: HTML collapses runs
            // of spaces, and a monospace font with collapsed spaces lines nothing up.
            rows.append("<div style=\"margin:3px 0;font-family:" + Ui.MONO
                    + ";font-size:13px;white-space:pre\">")
                .append("<span style=\"display:inline-block;width:9px;height:9px;border-radius:2px;")
                .append("background:").append(colour(s.kind(), index))
                .append(";margin-right:8px\"></span>")
                .append(Ui.esc(row(s)))
                .append("</div>");
        }
        String body =
            Ui.img(svg(slots, size), alt(title, slots, size))
            + Ui.line("<b>" + Ui.esc(title) + "</b> is " + size + " bytes.")
            + Ui.details("The exact offsets", rows.toString(), false)
            + (note.isEmpty() ? "" : Ui.small(Ui.prose(note)));
        return Ui.card(Ui.BLUE, "layout", body);
    }

    /**
     * One slot as a line of text, used by both renderings.
     *
     * One format string rather than two, because the terminal version and the list
     * behind the picture say the same thing and there is no version of this project
     * where it is good for them to drift.
     */
    static String row(Slot s) {
        return String.format("%-18s bytes %2d to %2d  (%d)",
            s.label(), s.offset(), s.offset() + s.width() - 1, s.width());
    }

    /** The same thing for a terminal, where there is no picture to look at. */
    static String text(String title, List<Slot> slots, long size) {
        StringBuilder out = new StringBuilder(title + " is " + size + " bytes\n\n");
        for (Slot s : slots) {
            out.append("  ").append(row(s)).append("\n");
        }
        return out.toString();
    }
}

// jvx is the small helper surface every lesson gets for free. It is deliberately thin.
// Anything it does that a reader could do themselves in three lines, it does in a way
// they can read, and it never hides the tool underneath it. When a lesson wants JOL or
// jcmd or jfr, the lesson calls JOL or jcmd or jfr, because watching the real tool is
// the point and a wrapper would be one more thing to trust.
//
// The class name is lower case. That is not a mistake and not Java style. It is a
// namespace that reads like one at a call site, `jvx.mark(o)`, and every lesson has it.

class jvx {

    static final String PIN = "jdk-27+35";
    static final String BUILT_FROM = "jvx/00-imports.jsh, jvx/05-ui.jsh, jvx/10-markword.jsh, jvx/15-gate.jsh, jvx/18-lens.jsh, jvx/20-jvx.jsh";

    // -- reading raw object memory ------------------------------------------------
    //
    // There is no supported API for reading the bytes of an object header. That is not
    // an oversight, it is the whole reason the mark word is an implementation detail:
    // the JVMS does not mandate any internal structure for objects at all (JVMS 2.7),
    // so there is nothing for an API to promise. Two internal doors are open on JDK 27
    // and jvx tries them in this order.
    //
    //   1. jdk.internal.misc.Unsafe, which needs
    //      --add-exports java.base/jdk.internal.misc=ALL-UNNAMED on the command line.
    //      Preferred, because it prints nothing.
    //   2. sun.misc.Unsafe, which needs no flags and prints four lines of terminal
    //      deprecation warning the first time. It still works on JDK 27 and it is on
    //      its way out.
    //
    // Which door opened is not hidden. jvx.markRoute() says, and the banner prints it,
    // because "where did this number come from" is a question a reader is entitled to
    // ask about a number that came from reading memory directly.

    private static Object unsafe;
    private static Method getLongMethod;
    private static Method fieldOffsetMethod;
    private static Method arrayBaseMethod;
    private static Method arrayScaleMethod;
    private static boolean fieldOffsetTakesAField;
    private static String markRoute = "not tried yet";

    private static void openUnsafe() {
        if (getLongMethod != null) return;
        try {
            Class<?> c = Class.forName("jdk.internal.misc.Unsafe");
            unsafe = c.getMethod("getUnsafe").invoke(null);
            getLongMethod = c.getMethod("getLong", Object.class, long.class);
            // This one names the field with a string. The older door wants a
            // reflected Field object instead, which is why the two are not
            // interchangeable and why fieldOffset below has to know which it got.
            fieldOffsetMethod = c.getMethod("objectFieldOffset", Class.class, String.class);
            arrayBaseMethod = c.getMethod("arrayBaseOffset", Class.class);
            arrayScaleMethod = c.getMethod("arrayIndexScale", Class.class);
            fieldOffsetTakesAField = false;
            markRoute = "jdk.internal.misc.Unsafe";
            return;
        } catch (Throwable ignored) {
            // Not exported to us. Fall through to the older door.
        }
        try {
            Class<?> c = Class.forName("sun.misc.Unsafe");
            Field f = c.getDeclaredField("theUnsafe");
            f.setAccessible(true);
            unsafe = f.get(null);
            getLongMethod = c.getMethod("getLong", Object.class, long.class);
            fieldOffsetMethod = c.getMethod("objectFieldOffset", Field.class);
            arrayBaseMethod = c.getMethod("arrayBaseOffset", Class.class);
            arrayScaleMethod = c.getMethod("arrayIndexScale", Class.class);
            fieldOffsetTakesAField = true;
            markRoute = "sun.misc.Unsafe (deprecated for removal, expect a warning)";
            return;
        } catch (Throwable t) {
            markRoute = "neither door opened: " + t;
            throw new UnsupportedOperationException(
                "cannot read object memory on this JVM. Start the kernel with "
                + "--add-exports java.base/jdk.internal.misc=ALL-UNNAMED. " + markRoute);
        }
    }

    static String markRoute() {
        openUnsafe();
        return markRoute;
    }

    /** The eight bytes at offset 0 of an object, exactly as they sit in memory. */
    static long mark(Object o) {
        openUnsafe();
        try {
            return (Long) getLongMethod.invoke(unsafe, o, 0L);
        } catch (Exception e) {
            throw new RuntimeException("reading the mark word failed", e);
        }
    }

    /** The mark word, printed with its fields separated out. */
    static void header(Object o) {
        System.out.print(MarkWord.decode(mark(o)));
    }

    static String hex(long word) {
        return MarkWord.hex(word);
    }

    static long field(Object o, String name) {
        return MarkWord.get(mark(o), name);
    }

    /**
     * The mark word of an object nothing has ever touched.
     *
     * This exists because of a trap that is very easy to fall into and impossible to
     * see. Assigning an object to a top level JShell variable is enough to make
     * something ask for its identity hash, and asking is what makes HotSpot write one
     * into the mark word. So this:
     *
     *     Object o = new Object();
     *
     * hands you an object whose hash field is already filled in, and the "before"
     * measurement you were about to take is gone. The semicolon does not save you.
     * Measured on 27+35-2325: a top level variable reads a nonzero hash with the
     * semicolon on the end, the same allocation stored into a static field of a class
     * declared in the same session reads 0, and one that is never named reads 0.
     *
     * Nothing here can touch the object between allocating it and reading it, because
     * the reference never leaves this method. It is the honest "before".
     */
    static long freshMark() {
        return mark(new Object());
    }

    /** Where every bit position jvx believes in came from. */
    static void provenance() {
        System.out.print(MarkWord.provenance());
    }

    // -- measuring layout ------------------------------------------------------------
    //
    // There is a library for this, JOL, and it is a good one. These four methods are
    // here anyway, because reaching for a download is the difference between a lesson
    // a reader can start in thirty seconds and one they cannot, and because the whole
    // trick fits in a sentence: the header is the thing your first field comes after,
    // so the offset of the first field is the size of the header. Nothing is being
    // hidden here. Read the four methods and you have the technique.

    /** The byte offset of one field within an instance. */
    static long fieldOffset(Class<?> owner, String name) {
        openUnsafe();
        try {
            if (fieldOffsetTakesAField) {
                Field f = owner.getDeclaredField(name);
                return (Long) fieldOffsetMethod.invoke(unsafe, f);
            }
            return (Long) fieldOffsetMethod.invoke(unsafe, owner, name);
        } catch (NoSuchFieldException e) {
            throw new IllegalArgumentException(owner.getName() + " has no field called " + name);
        } catch (Exception e) {
            throw new RuntimeException("could not read the offset of " + name, e);
        }
    }

    /**
     * Where the header stops, in bytes, for instances of this class.
     *
     * This is the offset of the earliest field, which is the same thing. A class with
     * no fields at all has no first field to point at, so it has to say so rather than
     * return a number it does not know.
     */
    static long headerSize(Class<?> type) {
        long earliest = Long.MAX_VALUE;
        for (Class<?> c = type; c != null; c = c.getSuperclass()) {
            for (Field f : c.getDeclaredFields()) {
                if (!Modifier.isStatic(f.getModifiers())) {
                    earliest = Math.min(earliest, fieldOffset(c, f.getName()));
                }
            }
        }
        if (earliest == Long.MAX_VALUE) {
            throw new IllegalArgumentException(
                type.getName() + " has no instance fields, so there is no first field offset "
                + "to measure the header with. Add one field and measure that class instead.");
        }
        return earliest;
    }

    /**
     * How wide a reference is, measured rather than assumed.
     *
     * One slot of an Object[] is one reference, so the array's index scale is the answer.
     * This is 4 with compressed oops and 8 without, and guessing it wrong throws every
     * object size in a lesson off by four bytes per field.
     */
    static long refWidth() {
        openUnsafe();
        try {
            return ((Number) arrayScaleMethod.invoke(unsafe, Object[].class)).longValue();
        } catch (Exception e) {
            throw new RuntimeException("could not read the reference width", e);
        }
    }

    /** How many bytes a field of this type takes up inside an object. */
    static long widthOf(Class<?> type) {
        if (type == long.class || type == double.class) return 8;
        if (type == int.class || type == float.class) return 4;
        if (type == short.class || type == char.class) return 2;
        if (type == byte.class || type == boolean.class) return 1;
        return refWidth();
    }

    /** Where an array's elements start, which is the size of an array header. */
    static long arrayBase(Class<?> arrayType) {
        openUnsafe();
        try {
            return ((Number) arrayBaseMethod.invoke(unsafe, arrayType)).longValue();
        } catch (Exception e) {
            throw new RuntimeException("could not read the array base offset", e);
        }
    }

    /**
     * Wrap a class declaration in a program that measures it, ready for jvx.run.
     *
     * The declaration has to be called Candidate. The launcher class has to come first
     * in the file and has to match the file name, which is how the single file source
     * launcher decides what to run, so the reader's class cannot be the first one.
     */
    static String sizeProbe(String candidateSource) {
        return """
            import jdk.internal.misc.Unsafe;
            import java.lang.reflect.Field;
            import java.lang.reflect.Modifier;

            public class Answer {
                public static void main(String[] args) {
                    Unsafe u = Unsafe.getUnsafe();
                    long first = Long.MAX_VALUE;
                    long end = 0;
                    for (Class<?> c = Candidate.class; c != null; c = c.getSuperclass()) {
                        for (Field f : c.getDeclaredFields()) {
                            if (Modifier.isStatic(f.getModifiers())) continue;
                            long off = u.objectFieldOffset(c, f.getName());
                            first = Math.min(first, off);
                            end = Math.max(end, off + width(u, f.getType()));
                        }
                    }
                    if (first == Long.MAX_VALUE) {
                        System.out.println("Candidate has no instance fields, so give it one");
                        return;
                    }
                    long size = (end + 7) / 8 * 8;
                    System.out.printf("header stops at %d, fields end at %d, object is %d bytes%n",
                        first, end, size);
                }

                static int width(Unsafe u, Class<?> t) {
                    if (t == long.class || t == double.class) return 8;
                    if (t == int.class || t == float.class) return 4;
                    if (t == short.class || t == char.class) return 2;
                    if (t == byte.class || t == boolean.class) return 1;
                    // A reference is as wide as one slot of an Object[], which is where
                    // compressed oops show up as 4 rather than 8.
                    return u.arrayIndexScale(Object[].class);
                }
            }

            """ + candidateSource + "\n";
    }

    // -- HeapLens, the object layout viewer ------------------------------------------
    //
    // Everything here is measured on the VM the reader is on. Nothing is looked up in a
    // table and nothing is assumed from the platform, because the whole lesson is that
    // the answer moved and the books have not caught up. The drawing is in Lens, which
    // is handed the measurements and never takes any.

    /**
     * A class with exactly one field, used as a ruler.
     *
     * A class with no instance fields has no first field to point at, so there is
     * nothing in it to measure the header with. Every non array object on HotSpot has
     * the same header, so measuring it on this one and saying so is better than either
     * refusing to draw `Object` or printing a number from a book.
     */
    private static class Ruler { byte b; }

    static long alignment() {
        String value = flag("ObjectAlignmentInBytes");
        return value == null ? 8L : Long.parseLong(value);
    }

    /** Where the header stops on this VM, for any object that has no fields to ask. */
    static long headerSize() {
        return fieldOffset(Ruler.class, "b");
    }

    /** Every instance field of this class and its superclasses, earliest first. */
    private static List<Field> instanceFields(Class<?> type) {
        List<Field> found = new ArrayList<>();
        for (Class<?> c = type; c != null; c = c.getSuperclass()) {
            for (Field f : c.getDeclaredFields()) {
                if (!Modifier.isStatic(f.getModifiers())) found.add(f);
            }
        }
        found.sort((a, b) -> Long.compare(
            fieldOffset(a.getDeclaringClass(), a.getName()),
            fieldOffset(b.getDeclaringClass(), b.getName())));
        return found;
    }

    /** How many bytes an instance takes, including the padding at the end. */
    static long sizeOf(Class<?> type) {
        long end = headerSize();
        for (Field f : instanceFields(type)) {
            end = Math.max(end,
                fieldOffset(f.getDeclaringClass(), f.getName()) + widthOf(f.getType()));
        }
        long align = alignment();
        return (end + align - 1) / align * align;
    }

    /**
     * The whole object as a list of byte runs, in order, with nothing left out.
     *
     * The gaps are the point. A field that does not start where the last one ended has
     * something in between, and that something is alignment padding the reader did not
     * ask for and is paying for. Naming it in the same list as the fields is what makes
     * it visible.
     */
    static List<Lens.Slot> layout(Class<?> type) {
        List<Lens.Slot> slots = new ArrayList<>();
        long cursor = headerSize();
        slots.add(new Lens.Slot("header", 0, cursor, "header"));
        for (Field f : instanceFields(type)) {
            long at = fieldOffset(f.getDeclaringClass(), f.getName());
            long width = widthOf(f.getType());
            if (at > cursor) {
                slots.add(new Lens.Slot("gap", cursor, at - cursor, "gap"));
            }
            slots.add(new Lens.Slot(
                f.getType().getSimpleName() + " " + f.getName(), at, width, "field"));
            cursor = at + width;
        }
        long size = sizeOf(type);
        if (size > cursor) {
            slots.add(new Lens.Slot("padding", cursor, size - cursor, "padding"));
        }
        return slots;
    }

    /** Draw one object's layout, byte by byte, as measured on this VM. */
    static void lens(Class<?> type) {
        List<Lens.Slot> slots = layout(type);
        long size = sizeOf(type);
        String title = type.getSimpleName();
        String note = "Measured on this VM, where `UseCompactObjectHeaders` is "
            + (flag("UseCompactObjectHeaders") == null ? "not a flag" : flag("UseCompactObjectHeaders"))
            + " and `ObjectAlignmentInBytes` is " + alignment() + ".";
        if (instanceFields(type).isEmpty()) {
            note = title + " has no instance fields, so the header was measured on a class "
                + "that has one. Every non array object on HotSpot has the same header. " + note;
        }
        if (Ui.html(Lens.card(title, slots, size, note))) return;
        System.out.print(Lens.text(title, slots, size));
        System.out.println();
        System.out.println(note.replace("`", ""));
    }

    // -- asking the VM about itself -----------------------------------------------
    //
    // This part needs no internal access at all. HotSpotDiagnosticMXBean is supported
    // API in the jdk.management module and it answers for any flag the VM has,
    // including the origin, which is the part people forget to check. A flag that is
    // true because it is the default and a flag that is true because somebody put it
    // on the command line are different facts, and a lesson that confuses them is
    // teaching a local accident as a general truth.

    private static HotSpotDiagnosticMXBean diagnostic() {
        return ManagementFactory.getPlatformMXBean(HotSpotDiagnosticMXBean.class);
    }

    /** A flag's value, or null when this VM has no such flag. */
    static String flag(String name) {
        try {
            return diagnostic().getVMOption(name).getValue();
        } catch (IllegalArgumentException e) {
            return null;
        }
    }

    /** Where a flag's value came from: default, command line, ergonomic, and so on. */
    static String flagOrigin(String name) {
        try {
            return diagnostic().getVMOption(name).getOrigin().toString();
        } catch (IllegalArgumentException e) {
            return null;
        }
    }

    static boolean on(String name) {
        return "true".equals(flag(name));
    }

    /**
     * A flag with its origin, the way `java -XX:+PrintFlagsFinal` would show it.
     *
     * Formatted into a string and printed once, rather than printf, and that is not a
     * style choice. The kernel turns every write on System.out into its own stream
     * message, and java.util.Formatter writes each padding space separately, so a
     * printf with a %-28s in it arrives at the reader as thirty little pieces and the
     * notebook renders each one on its own line. One string, one println, one line.
     */
    static void flags(String... names) {
        for (String name : names) {
            String value = flag(name);
            if (value == null) {
                System.out.println(String.format(
                    "%-28s %-10s %s", name, "-", "this VM has no such flag"));
            } else {
                System.out.println(String.format(
                    "%-28s %-10s {%s}", name, value, flagOrigin(name).toLowerCase()));
            }
        }
    }

    // -- running something in a different JVM -------------------------------------

    /**
     * Compile and run a single Java file in a fresh JVM, with the flags you give it,
     * and hand back everything it printed.
     *
     * Two quite different jobs need this and it is worth being clear about both.
     *
     * The first is that a lot of what this project teaches is only visible by
     * comparison, and the two things being compared are two JVMs started differently.
     * You cannot turn UseCompactObjectHeaders off in a running VM. The objects are
     * already laid out.
     *
     * The second is that the kernel you are typing into is a JShell, and JShell
     * changes some of what a lesson wants to observe. It wraps every snippet in a
     * synthetic class, which shows up in class histograms and compilation logs, and it
     * touches the objects you assign to variables. A subprocess has none of that,
     * because it is a plain JVM running a plain program.
     *
     * No javac step and no classpath, because a single .java file passed to `java` is
     * compiled in memory and run. That has been standard since JEP 330 in Java 11, and
     * it is why the source in a lesson cell is the whole program rather than the
     * interesting half of one.
     */
    static String run(String className, String source, String... vmArgs) {
        try {
            Path dir = Files.createTempDirectory("jvx");
            Path file = dir.resolve(className + ".java");
            Files.writeString(file, source);

            List<String> command = new ArrayList<>();
            command.add(Path.of(System.getProperty("java.home"), "bin", "java").toString());
            for (String arg : vmArgs) command.add(arg);
            command.add(file.toString());

            // Merged, and on purpose. A VM that refuses a flag says so on stderr, and a
            // reader who gets silence and no output has been told nothing at all.
            Process p = new ProcessBuilder(command).redirectErrorStream(true).start();
            String out = new String(p.getInputStream().readAllBytes());
            p.waitFor();

            Files.deleteIfExists(file);
            Files.deleteIfExists(dir);
            return out;
        } catch (Exception e) {
            throw new RuntimeException("could not run " + className + " in a fresh JVM", e);
        }
    }

    /** The same thing, printed rather than returned, which is what a cell usually wants. */
    static void show(String className, String source, String... vmArgs) {
        System.out.print(run(className, source, vmArgs));
    }

    // -- prediction gates -----------------------------------------------------------
    //
    // Three calls, forwarded to Gate. A lesson never names Gate, so the day the text
    // version is replaced by a widget, no lesson changes.

    /** Ask a question and stop. Nothing here shows the answer. */
    static void gate(String id, String question, String... options) {
        Gate.ask(id, question, options);
    }

    /** Write your answer down. It is not marked yet. */
    static void answer(String id, String choice) {
        Gate.answer(id, choice);
    }

    /** Mark it. Run this after you have measured, not before. */
    static void reveal(String id, String correct) {
        Gate.reveal(id, correct);
    }

    // -- what am I running on -----------------------------------------------------

    /**
     * Printed by the bootstrap cell of every lesson. It is not decoration. Almost every
     * observation in this project is true of one configuration and false of another, so
     * a reader comparing their output with the page needs to see, on the same screen,
     * which configuration produced theirs.
     */
    static void banner() {
        Runtime.Version v = Runtime.version();
        System.out.println(String.format("java      %s  (%s)",
            v, System.getProperty("java.vm.version")));
        System.out.println(String.format("vm        %s", System.getProperty("java.vm.name")));
        System.out.println(String.format("on        %s %s",
            System.getProperty("os.name"), System.getProperty("os.arch")));
        System.out.println(String.format("lessons pinned to %s", PIN));
        System.out.println();
        // Three flags, not four. UseCompressedClassPointers used to belong in this
        // list and no longer exists on JDK 27, which is exactly why flag() returns
        // null for a missing flag rather than throwing: a banner that dies because
        // the VM moved on is a banner that stops anyone from reading anything.
        flags("UseCompactObjectHeaders", "UseCompressedOops", "ObjectAlignmentInBytes");
        System.out.println();
        System.out.println("mark word read through " + markRoute());
        if (!v.toString().startsWith(PIN.replace("jdk-", "").split("\\+")[0])) {
            System.out.println();
            System.out.println("NOTE: this VM is not the pinned one. Numbers below may differ from the page,");
            System.out.println("      and where they do, this VM is right about this VM and the page is right");
            System.out.println("      about " + PIN + ".");
        }
    }
}

jvx.banner();


## The first question

Answer before you run anything. Wrong is fine and wrong is useful, but skipping is neither.

In [ ]:
jvx.gate("gate_1",
    "How many bytes does a bare `new Object()` occupy on JDK 27?",
    "a) 8",
    "b) 12",
    "c) 16",
    "d) it depends on the platform");

In [ ]:
// The letter here is a guess, and probably not yours. Change it, then run this cell.
jvx.answer("gate_1", "c");

### Measuring it, with nothing installed

There is a library for this called JOL, and reaching for it here would be a mistake, because the whole answer is available without it and the shape of the answer is the lesson.

The header is the thing your first field comes after. So the offset of the first field is where the header stops, which means it is the header size. That is the whole trick, and `jvx.headerSize` is four lines doing exactly it. The source is in the bootstrap cell above if you want to read it.

In [ ]:
// An object with no fields has no first field to point at, so measure classes that
// have exactly one and see where it starts.
class OneInt { int a; }
class OneRef { Object r; }

System.out.println("OneInt header stops at byte " + jvx.headerSize(OneInt.class));
System.out.println("OneRef header stops at byte " + jvx.headerSize(OneRef.class));
System.out.println("int[] elements start at byte " + jvx.arrayBase(int[].class));
System.out.println("long[] elements start at byte " + jvx.arrayBase(long[].class));

In [ ]:
jvx.reveal("gate_1", "a");

The header stops at byte 8. An object with no fields is 8 bytes, and that is already 8 byte aligned, so there is nothing to pad.

If you said 16, you were right until JDK 27 and you are in very good company, because that is what most books, most blog posts and most interview answers still say. If you said 12 you were remembering the header without the padding. If you said it depends on the platform, that is the most defensible wrong answer of the four, and it is worth two sentences.

It does depend on things, but not on the platform. Every number in this lesson was measured on both an arm64 laptop and an x86_64 Linux server, and every offset was identical. What it depends on is a flag, which you will turn off in a minute.

## What is actually in there

An object on the heap starts with a header, and the header exists because the runtime needs to know things about an object that the object's own fields cannot tell it. What class is this. Is anything holding a lock on it. Has anyone asked for its identity hash. How many collections has it survived.

The first 8 bytes are the mark word. It is not a number that means one thing, it is several unrelated fields packed into one word, and which fields are present depends on what has happened to the object. The lock state lives in the bottom two bits {[HOTSPOT src/hotspot/share/oops/markWord.hpp:124@jdk-27+35]}. The collector's age counter is four bits above that {[HOTSPOT src/hotspot/share/oops/markWord.hpp:126@jdk-27+35]}. The identity hash, if anything has ever asked for one, is 31 bits in the middle {[HOTSPOT src/hotspot/share/oops/markWord.hpp:128@jdk-27+35]}.

Then there is the class pointer, and this is the part that changed. Until JDK 27, it was a separate 4 byte field that came after the mark word, so the header was 8 plus 4 {[HOTSPOT src/hotspot/share/oops/markWord.hpp:43@jdk-27+35]}, and since almost every object then needs 4 bytes of padding to reach an 8 byte boundary, an empty object cost 16. From JDK 27, `UseCompactObjectHeaders` is on by default {[HOTSPOT src/hotspot/share/runtime/globals.hpp:131@jdk-27+35]} and the class pointer moved into the top 22 bits of the mark word itself {[HOTSPOT src/hotspot/share/oops/markWord.hpp:150@jdk-27+35]}. No separate field, no padding, header of 8.

Twenty two bits is the interesting constraint. It means a JVM can address about four million distinct classes, which is far more than any real program loads, and it is only possible because class metadata is allocated in one contiguous region so a class can be named by its offset into that region rather than by a full pointer.

### None of this is in the specification

It is worth being precise about what kind of fact you are learning here, because this is the single most common way people get burned by internals knowledge.

The Java Virtual Machine Specification says, in as many words, that "the Java Virtual Machine does not mandate any particular internal structure for objects" {[JVMS §2.7@SE25]}. Not the header size, not the mark word, not the class pointer, not the bit positions. All of it is HotSpot's choice, and OpenJ9 makes different ones.

So every claim in this lesson except that one sentence is a `[HOTSPOT]` claim, and every one of them cites a line of HotSpot source at the pinned tag. That is not bureaucracy. It is the difference between "objects are 8 bytes now" and "objects are 8 bytes on this implementation at this version with this flag", and only the second one is true.

### Why the flag exists at all

If compact headers are strictly better, why is there a switch? Because the change is not free. Twenty two bits caps the class space, and the identity hash lost a bit. More practically, code that reads object headers directly, and there is more of it in the wild than you would like, breaks. The flag is how you find out whether that is you.

You are going to use the flag as a measuring instrument rather than as a setting. Running the same program both ways, in two fresh JVMs, is the only way to see the difference, because a flag that decides object layout cannot be changed in a VM whose objects are already laid out.

## The picture

![The object header in bytes and in bits](https://raw.githubusercontent.com/tamnd/jvm-internals/main/docs/generated/markword.svg)

Nothing in that drawing was drawn by hand. The bit positions come from `markWord.hpp` at the pinned tag, and the picture is generated from them, so a field that moves in a future release moves in the picture rather than quietly making it wrong.

In [ ]:
// The same bits, on an object you made a moment ago. Use jvx.freshMark rather than
// allocating into a variable, because assigning an object to a top level variable in
// JShell is enough to make something ask for its identity hash, and then the "before"
// you wanted to look at is gone.
System.out.print(MarkWord.decode(jvx.freshMark()));

Look at the age field. It may read 0 and it may read 1, and both are right.

Which one you get depends on whether a young collection happened between the object being allocated and the word being read, which is not something the page can control. On the machine this was written on, the first run of that cell in a fresh kernel reports age 1 and every run after it reports 0, because that allocation is the one that fills eden, and the object survives the collection it triggered and comes back with its age counter incremented. Run the cell a second time and watch it drop to 0.

That is the age counter doing its job, and it is worth seeing this early: a header field is live state that the runtime writes as things happen to the object, not a constant stamped in at birth. You will see the same thing again with the identity hash further down, where it is easier to trigger on purpose.

## The second question

Now a class with two fields in it.

```java
class Two { int a; int b; }
```

In [ ]:
jvx.gate("gate_2",
    "How many bytes is one `Two`, with compact headers on and with them off?",
    "a) 16 compact, 16 legacy",
    "b) 16 compact, 20 legacy",
    "c) 16 compact, 24 legacy",
    "d) 12 compact, 16 legacy");

In [ ]:
// Same as before, the letter is a guess. Change it to yours.
jvx.answer("gate_2", "b");

### Running the same program in two different JVMs

`UseCompactObjectHeaders` decides where fields go, so it can only be chosen when the VM starts. To compare, you need two VMs. `jvx.run` writes a single Java file, hands it to `java`, and gives you back what it printed. There is no compile step and no classpath, because a lone `.java` file passed to `java` is compiled in memory and run.

In [ ]:
String probe = """
    import jdk.internal.misc.Unsafe;
    public class Size {
        static class Two { int a; int b; }
        public static void main(String[] a) {
            Unsafe u = Unsafe.getUnsafe();
            long first = u.objectFieldOffset(Two.class, "a");
            long last  = u.objectFieldOffset(Two.class, "b");
            long used  = last + 4;
            long size  = (used + 7) / 8 * 8;
            System.out.printf("header stops at %d, fields end at %d, object is %d bytes%n",
                first, used, size);
        }
    }
    """;

String open = "--add-exports=java.base/jdk.internal.misc=ALL-UNNAMED";
System.out.print("compact  " + jvx.run("Size", probe, open));
System.out.print("legacy   " + jvx.run("Size", probe, open, "-XX:-UseCompactObjectHeaders"));

In [ ]:
jvx.reveal("gate_2", "c");

Compact is 8 for the header plus 4 plus 4, which is 16 exactly and needs no padding. Legacy is 12 for the header plus 4 plus 4, which is 20, and 20 is not a multiple of 8, so it rounds up to 24.

If you said 20 you did the header arithmetic correctly and forgot the alignment, which is the most common way to be wrong here. Objects are allocated on 8 byte boundaries, controlled by `ObjectAlignmentInBytes`, so no object is ever an odd size.

So this one really did save 8 bytes per instance, not 4, which is the version of the story everybody repeats.

## The third question

`java.lang.Integer` is a class with one `int` field in it. Nothing else.

In [ ]:
jvx.gate("gate_3",
    "How many bytes is one `java.lang.Integer`, compact and legacy?",
    "a) 12 compact, 16 legacy",
    "b) 16 compact, 16 legacy",
    "c) 16 compact, 20 legacy",
    "d) 16 compact, 24 legacy");

In [ ]:
// Change the letter to yours before you run it.
jvx.answer("gate_3", "a");

In [ ]:
String probe3 = """
    import jdk.internal.misc.Unsafe;
    public class Boxed {
        public static void main(String[] a) throws Exception {
            Unsafe u = Unsafe.getUnsafe();
            long off = u.objectFieldOffset(Integer.class, "value");
            long used = off + 4;
            System.out.printf("header stops at %d, fields end at %d, object is %d bytes%n",
                off, used, (used + 7) / 8 * 8);
        }
    }
    """;

System.out.print("compact  " + jvx.run("Boxed", probe3, open));
System.out.print("legacy   " + jvx.run("Boxed", probe3, open, "-XX:-UseCompactObjectHeaders"));

In [ ]:
jvx.reveal("gate_3", "b");

16 bytes either way. The header shrank by 4 bytes and the object did not shrink at all.

Compact: 8 for the header plus 4 for the `int` is 12, which rounds up to 16. Legacy: 12 for the header plus 4 for the `int` is 16 exactly. The four bytes the header gave back went straight into padding, and the reader who was told "compact headers save four bytes per object" got nothing here.

This is the point of the lesson. That sentence is true about the header and it is not reliably true about the object, because every object is rounded up to a multiple of 8 anyway {[HOTSPOT src/hotspot/share/utilities/align.hpp:125@jdk-27+35]}. Whether you actually keep the saving depends on what your fields add up to. `Two` kept it, and kept twice as much as advertised. `Integer` kept none of it.

If your heap is mostly boxed numbers, compact headers may do nothing measurable for you. If it is mostly small objects with two or three fields, it can be a fifth of your heap. Neither of those is a guess you can make from the flag's description, and both of them are five minutes of measuring.

### Looking at one, byte by byte

Three numbers in a row is a hard way to see a layout. `jvx.lens` measures every field offset on the VM you are running and draws the object as bytes. Nothing in it is looked up: start this kernel with `-XX:-UseCompactObjectHeaders` and the picture changes.

In [ ]:
class Two { int a; int b; }
class Mixed { int a; long b; }
class Small { byte b; }
class Big extends Small { long x; }

jvx.lens(Two.class);
jvx.lens(Mixed.class);
jvx.lens(Big.class);

`Two` fills itself exactly, which is why it kept the whole 8 bytes. `Mixed` is not drawn in the order you declared it. The `long` comes first, at byte 8, because HotSpot lays the wide fields out first so each one lands on its own alignment, and the `int` takes what is left with 4 bytes of padding after it. Declaration order is not layout order.

`Big` is where the other colour comes from. `Small` has one `byte`, and `Big` cannot move it: code compiled against `Small` reads that field at that offset and knows nothing about `Big`. So the `long` waits for the next 8 byte boundary and bytes 9 to 15 belong to nobody. That is a gap rather than padding, because it sits in the middle rather than at the end. Add an `int` to `Big` and it lands at byte 12, inside the gap, for nothing.

## Proving the class pointer is really in there

The claim that the top 22 bits hold the class pointer is easy to state and easy to take on faith. It is also easy to check: two objects of different classes should differ in those bits, and two objects of the same class should not.

In [ ]:
// Two was declared a few cells up, where the lens drew it.
long objectKlass = MarkWord.get(jvx.freshMark(), "klass");
long twoKlassA = jvx.field(new Two(), "klass");
long twoKlassB = jvx.field(new Two(), "klass");

// Formatted, then printed. printf in a notebook cell arrives one fragment at a time and
// the front end puts each fragment on its own line, which is a thing worth knowing once.
System.out.println(String.format("Object     klass bits 0x%x", objectKlass));
System.out.println(String.format("Two        klass bits 0x%x", twoKlassA));
System.out.println(String.format("Two again  klass bits 0x%x", twoKlassB));
System.out.println();
System.out.println("two Twos agree:            " + (twoKlassA == twoKlassB));
System.out.println("Two differs from Object:   " + (twoKlassA != objectKlass));

Both true, so the field is doing what the header file says it does.

Do not write the actual number down anywhere. It is an offset into the region where class metadata lives, so it depends on what your program loaded and in what order. Measured on two machines, `Object` came out as `0x0017ac` on one and `0x001774` on the other. What is stable is the relationship, not the value.

## Watching a field get written

The identity hash is the clearest thing in the header to watch, because it starts empty and it is filled in lazily, the first time anybody asks. Nothing writes it at allocation.

In [ ]:
String probe4 = """
    import jdk.internal.misc.Unsafe;
    public class Hash {
        public static void main(String[] a) {
            Unsafe u = Unsafe.getUnsafe();
            Object o = new Object();
            System.out.printf("before  0x%016x%n", u.getLong(o, 0L));
            int h = System.identityHashCode(o);
            System.out.printf("after   0x%016x%n", u.getLong(o, 0L));
            System.out.printf("hash    0x%x, and bits 41..11 of the word say 0x%x%n",
                h, (u.getLong(o, 0L) >>> 11) & ((1L << 31) - 1));
        }
    }
    """;
jvx.show("Hash", probe4, open);

The word changes exactly once, in the middle, and the bits that changed decode to the number `identityHashCode` returned.

This runs in a separate JVM rather than here, for a reason worth knowing. JShell assigns your top level variables into its own machinery, and that is enough to make something ask the object for its identity hash. So an object you create in a cell and look at in the next cell already has a hash by the time you look. The "before" would be gone and you would never know it had been there. Anything that needs a genuinely untouched object goes through `jvx.run` or through `jvx.freshMark`.

In [ ]:
// Every bit position used above, and the line of HotSpot it came from.
jvx.provenance();

## The boss fight

Write a class called `Candidate` whose instances are exactly 24 bytes with compact headers on and exactly 32 bytes with them off, using only `int`, `long` and reference fields, and no more than four of them.

The two sizes have to differ by 8 rather than by 4, and the header change on its own only gives you 4. So you need the alignment padding to work against you in one configuration and not the other, which means thinking about where an 8 byte field is allowed to sit.

Measure, do not reason. Edit the cell until both numbers are right, then save your class to `lessons/O01/answer.java` and run `python lessons/O01/grade.py lessons/O01/answer.java`.

In [ ]:
String mine = """
    class Candidate {
        // your fields here
    }
    """;

// The same measurement the grader does, so you can iterate before you submit.
System.out.print("compact  " + jvx.run("Answer", jvx.sizeProbe(mine), open));
System.out.print("legacy   " + jvx.run("Answer", jvx.sizeProbe(mine), open, "-XX:-UseCompactObjectHeaders"));

## What this contributes to BP-HEADER

Three clauses, each of which is now measured rather than asserted.

**H1.** With `UseCompactObjectHeaders` on, which is the default from JDK 27, the object header is 8 bytes and contains the compressed class pointer in bits 63 to 42 of the mark word {[HOTSPOT src/hotspot/share/oops/markWord.hpp:150@jdk-27+35]}. With it off, the header is 12 bytes: an 8 byte mark word followed by a separate 4 byte class pointer.

**H2.** Instance size is the first field offset plus the space the fields occupy, rounded up to `ObjectAlignmentInBytes`, which defaults to 8. The rounding is why a 4 byte reduction in header size does not imply a 4 byte reduction in object size, and for `java.lang.Integer` it implies none at all.

**H3.** None of the above is specified. The JVMS does not mandate any internal structure for objects {[JVMS §2.7@SE25]}, so every clause here is a statement about HotSpot at `jdk-27+35` and about nothing else.

## What you now know

- You can measure the header size of any class on any JVM with one call and nothing installed, because the first field offset is where the header ends.
- You can predict an object's size from its fields, and you know to round up to 8 before you believe the answer.
- You can tell someone why compact object headers saved 8 bytes on one of your classes and 0 on another, without hand waving about it depending.
- You can read a mark word, name each field, and cite the line of HotSpot that puts it there.
- You know that all of this is HotSpot's choice rather than the specification's, and you know the sentence in the JVMS that says so.
- You know that JShell touches the objects you assign to variables, and which of your measurements that would quietly ruin.